# 02 - Data Validation (Phase 3)

Domain-specific validation on top of the structural audit from Phase 2
(`01_data_audit.ipynb`). All logic lives in `data_contracts/` and
`src/promolift/validation/` -- this notebook only calls into it and
reports results.

Two independent checks:
- **Referential integrity** -- is the raw data itself trustworthy? (schema
  contracts, client/product foreign keys, transaction date integrity)
- **Experiment validity** -- is the randomized experiment behind
  `uplift_train.csv` trustworthy? (treatment/control balance, outcome
  distribution, a raw ATE sanity check)

If either of these fails, no causal model built later can be trusted --
this notebook is the gate before Phase 5+ (modeling).

In [ ]:
import pandera.errors

from promolift.data.loader import Dataset
from promolift.validation.experiment_validity import (
    categorical_covariate_balance,
    numeric_covariate_balance,
    outcome_distribution,
    raw_ate_sanity_check,
)
from promolift.validation.referential_integrity import (
    client_foreign_key_integrity,
    product_foreign_key_integrity,
    transaction_date_integrity,
)
from promolift.validation.schema_validation import validate_dataset

## 1. Schema contracts

Validates every small file eagerly (full dtype + value checks) and
`purchases.csv` lazily (dtype checks only -- see
`schema_validation.validate_dataset` docstring for why value-level checks
aren't reliable on a Polars `LazyFrame`).

In [ ]:
for dataset in Dataset:
    try:
        result = validate_dataset(dataset)
        if dataset is Dataset.PURCHASES:
            result.collect()
        print(f"{dataset.value}: PASSED")
    except pandera.errors.SchemaError as e:
        print(f"{dataset.value}: FAILED\n{e}\n")

`clients` is expected to fail here: 313 of 400,162 rows have an
implausible `age` (min -7491, max 1901). This is a known, real data-quality
issue -- flagged for handling in feature engineering (Phase 5), not fixed
here.

## 2. Referential integrity

In [ ]:
print(client_foreign_key_integrity())
print(product_foreign_key_integrity())
print(transaction_date_integrity())

## 3. Experiment validity

Checks whether the treatment/control split in `uplift_train.csv` behaves
like a genuine randomization, and whether the observed effect is real.

In [ ]:
print(numeric_covariate_balance("age"))
print(categorical_covariate_balance("gender"))
print(outcome_distribution())
print(raw_ate_sanity_check())

## Phase 3 summary

- **Referential integrity: clean.** 0 orphaned `client_id`s or `product_id`s
  in `purchases.csv`; all `transaction_datetime` values parse and fall in a
  sane range (Nov 2018 - Mar 2019), no future dates.
- **Experiment validity: clean.** Age (SMD ~0.005) and gender (max
  proportion diff ~0.002) are well balanced between arms, far under the 0.1
  rule-of-thumb threshold. The raw ATE is +3.3pp with a 95% CI that excludes
  zero -- a real, non-degenerate effect.
- **One known issue, not fixed here:** 313 clients have an implausible age.
  Handle by filtering or clipping in feature engineering (Phase 5), not by
  changing the schema contract to accept it.

**Conclusion:** the dataset and the experiment behind it are trustworthy
enough to build causal models on top of (Phase 5+).